In [ ]:
# ==========================================
# Célula 1 - Configurações, Importações e Setup
# ==========================================
import os
import time
import glob
import shutil
import stat
import pandas as pd
from datetime import datetime, timedelta

# Selenium e Webdriver
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Lógica de Datas D-1
hoje = datetime.now()
ontem = hoje - timedelta(days=1)
primeiro_dia = ontem.replace(day=1, hour=0, minute=0, second=0, microsecond=0)

data_de_str = primeiro_dia.strftime("%d/%m/%Y")
data_ate_str = ontem.strftime("%d/%m/%Y")
pasta_mes = ontem.strftime("%Y%m")

caminho_base = os.getcwd()

# Mapeamento Multi-Tenant
tenants = [
    {
        "id": "27",
        "nome_exibicao": "27 - Brasil",
        "nome_empresa_mktzap": "27 - Natura - CN",
        "pasta_destino": os.path.abspath(os.path.join(caminho_base, "documentos", pasta_mes, "27 - Brasil")),
        "pasta_temp": os.path.abspath(os.path.join(caminho_base, "downloads_temp_27"))
    },
    {
        "id": "1531",
        "nome_exibicao": "1531 - Hispana",
        "nome_empresa_mktzap": "1531 - Natura - Experiencia CN OIs",
        "pasta_destino": os.path.abspath(os.path.join(caminho_base, "documentos", pasta_mes, "1531 - Hispana")),
        "pasta_temp": os.path.abspath(os.path.join(caminho_base, "downloads_temp_1531"))
    }
]

# Garantia da existência das pastas físicas
for t in tenants:
    os.makedirs(t["pasta_temp"], exist_ok=True)
    os.makedirs(t["pasta_destino"], exist_ok=True)
    
print("✅ Setup concluído com sucesso!")

In [ ]:
# ==========================================
# Célula 2 - Inicialização do Chrome e Login
# ==========================================
MKTZAP_URL_LOGIN = "https://natura.mktzap.com.br/login"

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-popup-blocking")
chrome_options.add_argument("--disable-features=DownloadBubble,DownloadBubbleV2")

prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "profile.default_content_settings.popups": 0
}
chrome_options.add_experimental_option("prefs", prefs)

print("🚀 Iniciando navegador do Selenium...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
wait_longo = WebDriverWait(driver, 600)  # Aguarda login manual
wait_curto = WebDriverWait(driver, 20)

driver.get(MKTZAP_URL_LOGIN)
print("⏳ Aguardando você realizar o login no MKTZap na janela aberta...")

wait_longo.until(EC.url_contains("/chat"))
print("✅ Login confirmado! O navegador permanecerá aberto para as próximas células.")

In [ ]:
# ==========================================
# Célula 3 - Funções Auxiliares (CDP, Troca de Empresa e Downloads)
# ==========================================
def definir_pasta_download(pasta_temp):
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": pasta_temp
    })

def trocar_empresa(nome_empresa):
    print(f"🏢 Alternando para a empresa: '{nome_empresa}'...")
    
    # Redireciona para o /chat antes de trocar a empresa para garantir que o menu lateral esteja carregado
    if "/chat" not in driver.current_url:
        driver.get("https://natura.mktzap.com.br/chat")
        time.sleep(2)

    menu_user = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-dropdown']")))
    driver.execute_script("arguments[0].scrollIntoView(true);", menu_user)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", menu_user)
    time.sleep(1)
    
    btn_alterar = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-a-change-company']")))
    driver.execute_script("arguments[0].click();", btn_alterar)
    time.sleep(1)
    
    input_empresa = wait_curto.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-bottom-modal-change-company-text-name']")))
    input_empresa.clear()
    
    for letra in nome_empresa:
        input_empresa.send_keys(letra)
        time.sleep(0.03)
    time.sleep(1.5)
    
    item_lista = wait_curto.until(EC.presence_of_element_located((By.XPATH, f"//li[contains(text(), '{nome_empresa}')]")))
    driver.execute_script("arguments[0].click();", item_lista)
    time.sleep(1)
    
    btn_trocar = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='menu-user-botton-modal-change-company-button-ok']")))
    driver.execute_script("arguments[0].click();", btn_trocar)
    time.sleep(5)

def aguardar_e_mover_download(pasta_temp, pasta_destino, prefixo_arquivo):
    inicio_espera = time.time()
    arquivo_baixado = None
    
    while time.time() - inicio_espera < 180: # 3 minutos
        arquivos = [f for f in glob.glob(os.path.join(pasta_temp, "*.*")) if not f.endswith('.crdownload') and not f.endswith('.tmp')]
        if arquivos:
            arquivo_baixado = max(arquivos, key=os.path.getmtime)
            if os.path.getmtime(arquivo_baixado) >= inicio_espera - 10:
                break
        time.sleep(3)
        
    if arquivo_baixado:
        ext = arquivo_baixado.split('.')[-1]
        nome_final = f"{prefixo_arquivo}_{primeiro_dia.strftime('%Y%m%d')}_a_{ontem.strftime('%Y%m%d')}.{ext}"
        caminho_final = os.path.join(pasta_destino, nome_final)
        
        if os.path.exists(caminho_final):
            os.remove(caminho_final)
            
        shutil.move(arquivo_baixado, caminho_final)
        print(f"  -> ✅ Arquivo salvo em: {nome_final}")
    else:
        print(f"  -> ⚠️ Timeout: O arquivo do relatório não apareceu a tempo na pasta de downloads.")

print("✅ Funções auxiliares atualizadas!")

In [ ]:
# ==========================================
# Célula 4 - Extração do Relatório Analítico (Background via CSV)
# ==========================================
URL_ANALITICO = "https://natura.mktzap.com.br/reports/analytic"
URL_BACKGROUND = "https://natura.mktzap.com.br/new/reports/background-reports"

# Definição dinâmica do atalho com base no dia atual:
texto_atalho_data = "Mês passado" if hoje.day == 1 else "Este mês"

for t in tenants:
    print(f"\n==========================================")
    print(f"📊 Processando Analítico: {t['nome_exibicao']}")
    print(f"==========================================")
    
    definir_pasta_download(t["pasta_temp"])
    trocar_empresa(t["nome_empresa_mktzap"])
    
    driver.get(URL_ANALITICO)
    time.sleep(3)
    
    # 1. Clica no atalho de data ("Este mês" ou "Mês passado")
    print(f"  -> Clicando no atalho de data: '{texto_atalho_data}'...")
    opcao_data = wait_curto.until(
        EC.element_to_be_clickable((By.XPATH, f"//a[contains(text(), '{texto_atalho_data}')]"))
    )
    driver.execute_script("arguments[0].click();", opcao_data)
    time.sleep(1.5)
    
    # 2. Garante/Garante a seleção do Formato CSV (Padrão)
    try:
        select_export = Select(wait_curto.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[ng-model='vm.exportAs']"))
        ))
        select_export.select_by_value("csv")
        print("  -> Formato CSV selecionado.")
    except Exception as e:
        print(f"  -> Aviso ao selecionar formato CSV: {e}")
    time.sleep(1)
    
    # 3. Clica no botão "Gerar Relatório"
    print("  -> Clicando em 'Gerar Relatório'...")
    btn_gerar = wait_curto.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_analytic_report_generate']"))
    )
    driver.execute_script("arguments[0].click();", btn_gerar)
    time.sleep(2)
    
    # Confirma modal de segundo plano, se exibido
    modais = driver.find_elements(By.XPATH, "//button[contains(text(),'Confirmar')]")
    if modais and modais[0].is_displayed():
        driver.execute_script("arguments[0].click();", modais[0])
        
    # 4. Navega para a tela de Relatórios em Background
    print("  -> Redirecionando para Relatórios em Background...")
    driver.get(URL_BACKGROUND)
    time.sleep(3)
    
    # 5. Monitora a tabela até o status mudar para "Finalizado"
    print("  -> Aguardando status mudar para 'Finalizado'...")
    inicio_espera = time.time()
    
    while time.time() - inicio_espera < 300: # Limite de 5 minutos
        try:
            primeira_linha = driver.find_element(By.XPATH, "//table//tbody//tr[1]")
            if "Finalizado" in primeira_linha.text:
                print("  -> Status 'Finalizado' detectado! Efetuando download...")
                btn_dl = primeira_linha.find_element(By.XPATH, ".//button[@title='Download' or contains(@class, 'btn-outline-primary')]")
                driver.execute_script("arguments[0].click();", btn_dl)
                break
        except Exception:
            pass
        time.sleep(5)
        driver.refresh()
        time.sleep(3)
        
    # 6. Escuta a pasta e move o arquivo baixado
    aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_analitico")

In [ ]:
## ==========================================
## Célula 5 - Extração do Relatório Completo
## ==========================================
#URL_COMPLETO = "https://natura.mktzap.com.br/report-bi/report-complete"
#
#for t in tenants:
#    print(f"\n==========================================")
#    print(f"📊 Processando Completo: {t['nome_exibicao']}")
#    print(f"==========================================")
#    
#    definir_pasta_download(t["pasta_temp"])
#    trocar_empresa(t["nome_empresa_mktzap"])
#    
#    driver.get(URL_COMPLETO)
#    time.sleep(3)
#    
#    input_from = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
#    input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
#    driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, data_de_str)
#    driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, data_ate_str)
#    driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
#    driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
#    
#    btn_gerar = driver.find_element(By.CSS_SELECTOR, "button[ga-event*='report_generate']")
#    driver.execute_script("arguments[0].click();", btn_gerar)
#    
#    aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_completo")

In [ ]:
# ==========================================
# Célula 6 - Extração do Relatório de Enquetes
# ==========================================
URL_ENQUETES = "https://natura.mktzap.com.br/new/reports/polls"

# Inputs do tipo "date" exigem a formatação ISO: YYYY-MM-DD
data_de_iso = primeiro_dia.strftime("%Y-%m-%d")
data_ate_iso = ontem.strftime("%Y-%m-%d")

for t in tenants:
    print(f"\n==========================================")
    print(f"📊 Processando Enquetes: {t['nome_exibicao']}")
    print(f"==========================================")
    
    definir_pasta_download(t["pasta_temp"])
    trocar_empresa(t["nome_empresa_mktzap"])
    
    driver.get(URL_ENQUETES)
    time.sleep(3)
    
    # 1. Injeta o período D-1 nos campos #polls-from e #polls-to
    print(f"  -> Injetando período D-1 ({data_de_iso} até {data_ate_iso})...")
    input_from = wait_curto.until(EC.presence_of_element_located((By.ID, "polls-from")))
    input_to = driver.find_element(By.ID, "polls-to")
    
    driver.execute_script(
        "arguments[0].value = arguments[1]; "
        "arguments[0].dispatchEvent(new Event('change')); "
        "if (window.angular) { angular.element(arguments[0]).triggerHandler('change'); }",
        input_from, data_de_iso
    )
    driver.execute_script(
        "arguments[0].value = arguments[1]; "
        "arguments[0].dispatchEvent(new Event('change')); "
        "if (window.angular) { angular.element(arguments[0]).triggerHandler('change'); }",
        input_to, data_ate_iso
    )
    time.sleep(1)
    
    # 2. Seleciona a Enquete no Dropdown #polls-poll
    dropdown_poll = Select(wait_curto.until(EC.presence_of_element_located((By.ID, "polls-poll"))))
    
    if t["id"] == "27":
        try:
            dropdown_poll.select_by_visible_text("C-SAT (Experiência CNF)")
            print("  -> Enquete 'C-SAT (Experiência CNF)' selecionada.")
        except Exception:
            dropdown_poll.select_by_index(1)
    else:
        try:
            dropdown_poll.select_by_visible_text("C-SAt - Satisfaccion")
            print("  -> Enquete 'C-SAt - Satisfaccion' selecionada.")
        except Exception:
            dropdown_poll.select_by_index(1)
            
    time.sleep(2)
    
    # 3. Clica em "Baixar detalhes da enquete"
    print("  -> Clicando em 'Baixar detalhes da enquete'...")
    btn_baixar = driver.find_element(By.XPATH, "//button[contains(., 'Baixar detalhes da enquete')]")
    driver.execute_script("arguments[0].click();", btn_baixar)
    
    # 4. Aguarda o download e move para a pasta da filial
    aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_enquetes")

In [ ]:
# ==========================================
# Célula 7 - Extração do Relatório de Tickets
# ==========================================
URL_TICKETS = "https://natura.mktzap.com.br/reports/history-tickets"

for t in tenants:
    print(f"\n==========================================")
    print(f"📊 Processando Tickets: {t['nome_exibicao']}")
    print(f"==========================================")
    
    definir_pasta_download(t["pasta_temp"])
    trocar_empresa(t["nome_empresa_mktzap"])
    
    driver.get(URL_TICKETS)
    time.sleep(3)
    
    input_from = wait_curto.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtFromDate']")))
    input_to = driver.find_element(By.CSS_SELECTOR, "input[ng-model='vm.filters.createdAtToDate']")
    driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_from, data_de_str)
    driver.execute_script("arguments[0].removeAttribute('readonly'); arguments[0].value = arguments[1];", input_to, data_ate_str)
    driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_from)
    driver.execute_script("angular.element(arguments[0]).triggerHandler('change');", input_to)
    
    btn_gerar = driver.find_element(By.CSS_SELECTOR, "button[ga-event='mktzap_bt_action_history_tickets_report_generate']")
    driver.execute_script("arguments[0].click();", btn_gerar)
    
    aguardar_e_mover_download(t["pasta_temp"], t["pasta_destino"], "bruto_tickets")

In [ ]:
# ==========================================
# Célula 8 - Encerrar a Sessão do Selenium
# ==========================================
driver.quit()
print("🔒 Sessão do Selenium finalizada com sucesso! Todos os arquivos brutos foram extraídos.")

In [ ]:
# ==========================================
# Célula 9 - Transformação Camada Gold
# ==========================================
import os
import glob
import html
import pandas as pd

def aplicar_regra_qtd(val):
    if pd.isna(val):
        return "1"
    val_str = str(val).strip()
    if val_str == "" or val_str.lower() in ["none", "nan", "null"]:
        return "1"
    try:
        num = float(val_str.replace(',', '.'))
        if num <= 0 or num >= 100:
            return "1"
        return str(int(num))
    except (ValueError, TypeError):
        return "1"

def desformatar_html(val):
    """Converte entidades HTML (ex: &#227;) em caracteres acentuados reais (ex: ã)."""
    if isinstance(val, str) and "&#" in val:
        return html.unescape(val)
    return val

def encontrar_linha_cabecalho_real(caminho):
    encodings = ['utf-8-sig', 'latin1', 'utf-8', 'utf-16']
    for enc in encodings:
        try:
            with open(caminho, 'r', encoding=enc) as f:
                for idx, line in enumerate(f):
                    line_lower = line.lower()
                    if 'protocolo' in line_lower or 'protocol' in line_lower or 'data contato' in line_lower:
                        return idx
        except Exception:
            pass
    return 0

def carregar_csv_flexivel(caminho, rel_tipo=""):
    if rel_tipo == "tickets":
        skip_count = 1
    else:
        skip_count = encontrar_linha_cabecalho_real(caminho)
        
    print(f"  -> Ignorando {skip_count} linha(s) de metadados do topo do arquivo...")
    
    for sep in [';', ',', '\t']:
        for enc in ['utf-8-sig', 'latin1', 'utf-8', 'utf-16']:
            try:
                df = pd.read_csv(
                    caminho, 
                    skiprows=skip_count, 
                    sep=sep, 
                    encoding=enc, 
                    on_bad_lines='skip', 
                    low_memory=False
                )
                if len(df.columns) > 3:
                    return df
            except Exception:
                pass
                
    return pd.read_csv(caminho, skiprows=skip_count, sep=None, engine='python', encoding='utf-8-sig')

relatorios_tipos = ["analitico", "completo", "enquetes", "tickets"]

for t in tenants:
    pasta_destino = t["pasta_destino"]
    
    for rel in relatorios_tipos:
        arquivos_brutos = glob.glob(os.path.join(pasta_destino, f"bruto_{rel}_*.*"))
        if arquivos_brutos:
            caminho_bruto = max(arquivos_brutos, key=os.path.getmtime)
            print(f"⏳ Processando Camada Gold do {rel.capitalize()} ({t['nome_exibicao']})...")
            
            try:
                # 1. Carga do CSV ignorando os metadados do topo
                df = carregar_csv_flexivel(caminho_bruto, rel)
                
                # 2. Limpeza dos nomes das colunas
                df.columns = df.columns.astype(str).str.strip()
                
                # 3. Correção de Caracteres Especiais / Entidades HTML
                for col in df.select_dtypes(include=['object', 'string']).columns:
                    df[col] = df[col].astype(str).apply(desformatar_html)
                
                # 4. Aplicação da Regra de Negócio: Qtd Atendimentos e Preservação do dado RAW
                cols_qtd = [c for c in df.columns if 'qtd' in c.lower() and 'atend' in c.lower()]
                if cols_qtd:
                    col_alvo = cols_qtd[0]
                    df['Qtd Atendimentos RAW'] = df[col_alvo].astype(str)
                    df['Qtd Atendimentos'] = df[col_alvo].apply(aplicar_regra_qtd)
                    if col_alvo not in ['Qtd Atendimentos', 'Qtd Atendimentos RAW']:
                        df.drop(columns=[col_alvo], inplace=True, errors='ignore')
                
                # 5. Higienização geral de strings e valores nulos
                for col in df.select_dtypes(include=['object', 'string']).columns:
                    df[col] = df[col].astype(str).str.strip().replace({'nan': '', 'None': '', 'NaN': ''})
                
                # 6. Exportação Gold oficial com codificação UTF-8 com BOM
                nome_gold = f"{hoje.strftime('%Y-%m-%d')}-relatorio-gold-{rel}-periodo-{primeiro_dia.strftime('%Y-%m-%d')}-a-{ontem.strftime('%Y-%m-%d')}.csv"
                caminho_gold = os.path.join(pasta_destino, nome_gold)
                
                df.to_csv(caminho_gold, sep=';', index=False, encoding='utf-8-sig')
                
                # 7. Expurgo da base bruta
                os.remove(caminho_bruto)
                print(f"  -> ✅ Relatório Gold gerado com sucesso: {nome_gold}")
                
            except Exception as e:
                print(f"  -> ❌ Erro ao converter {rel.capitalize()}: {str(e)}")

In [ ]:
# ==========================================
# Célula 10 - Limpeza das Pastas Temporárias do Sistema Operacional
# ==========================================
time.sleep(2)

def forcar_exclusao(func, path, exc_info):
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"⚠️ Aviso ao excluir {path}: {e}")

pastas_para_remover = ["downloads_temp_27", "downloads_temp_1531"]

for pasta in pastas_para_remover:
    caminho_pasta = os.path.join(caminho_base, pasta)
    if os.path.exists(caminho_pasta):
        try:
            shutil.rmtree(caminho_pasta, onerror=forcar_exclusao)
            print(f"✅ Pasta limpa: {pasta}")
        except Exception as e:
            print(f"⚠️ Pasta trancada no SO/OneDrive: {e}")

print("✨ Pipeline completo finalizado com sucesso!")